In [43]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten

In [44]:
df=pd.read_csv('training.csv')
df.head(10)

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3
5,ive been feeling a little burdened lately wasn...,0
6,ive been taking or milligrams or times recomme...,5
7,i feel as confused about life as a teenager or...,4
8,i have been with petronas for years i feel tha...,1
9,i feel romantic too,2


In [45]:
df.tail()

,text,label
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,1
15998,i feel like this was such a rude comment and i...,3
15999,i know a lot but i feel so stupid because i ca...,0


In [46]:
df.describe()

,label
count,16000.000000
mean,1.565937
std,1.501430
min,0.000000
25%,0.000000
50%,1.000000
75%,3.000000
max,5.000000


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    16000 non-null  object
 1   label   16000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 250.1+ KB


In [48]:
df.shape

(16000, 2)

In [49]:
df.fillna(0,inplace=True)

In [50]:
df['text'][1]

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [51]:
import re
print(df.iloc[2].text)

im grabbing a minute to post i feel greedy wrong


In [52]:
clean=re.compile('<.*?>')
print(re.sub(clean,'',df.iloc[1].text))

i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake


In [53]:
def clean_html(text):
  clean=re.compile('<.*?>')
  return re.sub(clean,'',text)

In [54]:
df['text']=df['text'].apply(clean_html)

In [55]:
df.head()

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


In [56]:
def remove_special(text):
  x=''
  for i in text:
    if i.isalnum():
      x=x+i
    else:
      x=x+' '
  return x

In [57]:
remove_special("Th%e @ classic use of the word. it is called oz as that is the nickname given to the oswald maximum")

'Th e   classic use of the word  it is called oz as that is the nickname given to the oswald maximum'

In [58]:
df['text']=df['text'].apply(remove_special)

In [59]:
df.head()

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


In [60]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [61]:
from nltk.corpus import stopwords
stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [62]:
def remove_stopwords(text):
  x=[]
  for i in text.split():
    if i not in stopwords.words('english'):
      x.append(i)
  y=x[:]
  x.clear()
  return y

In [63]:
df['text']=df['text'].apply(remove_stopwords)

In [64]:
print (df.head())

                                                text  label
0                          [didnt, feel, humiliated]      0
1  [go, feeling, hopeless, damned, hopeful, aroun...      0
2  [im, grabbing, minute, post, feel, greedy, wrong]      3
3  [ever, feeling, nostalgic, fireplace, know, st...      2
4                                 [feeling, grouchy]      3


In [65]:
y=[]
from nltk.stem import PorterStemmer
ps=PorterStemmer()
def stem_words(text):
  for i in text:
    y.append(ps.stem(i))
  z=y[:]
  y.clear()
  return z

In [66]:
df['text']=df['text'].apply(stem_words)

In [67]:
df.head()

,text,label
0,"[didnt, feel, humili]",0
1,"[go, feel, hopeless, damn, hope, around, someo...",0
2,"[im, grab, minut, post, feel, greedi, wrong]",3
3,"[ever, feel, nostalg, fireplac, know, still, p...",2
4,"[feel, grouchi]",3


In [68]:
def join_back(list_input):
  return " ".join(list_input)

In [69]:
df['text']=df['text'].apply(join_back)

In [70]:
df.head()

,text,label
0,didnt feel humili,0
1,go feel hopeless damn hope around someon care ...,0
2,im grab minut post feel greedi wrong,3
3,ever feel nostalg fireplac know still properti,2
4,feel grouchi,3


In [71]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000)

In [72]:
X=cv.fit_transform(df['text']).toarray()

In [73]:
X[1:5,:]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [74]:
X.shape

(16000, 5000)

In [75]:
y=df.iloc[:,-1].values

In [76]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,y,test_size=0.2)

In [77]:
model=Sequential()

In [78]:
model.add(Conv1D(filters=32,kernel_size=2, activation='relu',input_shape=(X_train.shape[1],1)))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [79]:
model.add(MaxPooling1D(pool_size=2))

In [80]:
model.add(Flatten())

In [81]:
from tensorflow.keras.layers import Dense
model.add(Dense(16,activation='relu'))

In [82]:
model.add(Dense(6,activation='softmax'))

In [83]:
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [84]:
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential

# One-hot encode the target variable
Y_train_encoded = to_categorical(Y_train, num_classes=6)
Y_test_encoded = to_categorical(Y_test, num_classes=6)

# Reshape X for LSTM input (samples, timesteps, features)
# Treating CountVectorizer output as a single timestep with X_train.shape[1] features.
X_train_reshaped = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_reshaped = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

# Define the LSTM model
lstm_model = Sequential()
lstm_model.add(LSTM(128, activation='relu', input_shape=(X_train_reshaped.shape[1], X_train_reshaped.shape[2])))
lstm_model.add(Dense(64, activation='relu'))
lstm_model.add(Dense(6, activation='softmax')) # 6 classes

# Compile the model
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("LSTM model summary:")
lstm_model.summary()

# Train the model
print("\nTraining LSTM model...")
history = lstm_model.fit(X_train_reshaped, Y_train_encoded, epochs=50, batch_size=32, validation_split=0.2)

# Evaluate the model
loss, accuracy = lstm_model.evaluate(X_test_reshaped, Y_test_encoded)
print(f"\nLSTM Model Test Accuracy: {accuracy*100:.2f}%")

LSTM model summary:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 128)            │     2,626,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,634,694 (10.05 MB)

 Trainable params: 2,634,694 (10.05 MB)

 Non-trainable params: 0 (0.00 B)


Training LSTM model...
Epoch 1/50
320/320 ━━━━━━━━━━━━━━━━━━━━ 15s 40ms/step - accuracy: 0.5832 - loss: 1.1473 - val_accuracy: 0.8246 - val_loss: 0.5604
Epoch 2/50
320/320 ━━━━━━━━━━━━━━━━━━━━ 20s 39ms/step - accuracy: 0.8988 - loss: 0.3149 - val_accuracy: 0.8480 - val_loss: 0.4678
Epoch 3/50
320/320 ━━━━━━━━━━━━━━━━━━━━ 13s 40ms/step - accuracy: 0.9543 - loss: 0.1366 - val_accuracy: 0.8406 - val_loss: 0.5477
Epoch 4/50
320/320 ━━━━━━━━━━━━━━━━━━━━ 13s 40ms/step - accuracy: 0.9751 - loss: 0.0823 - val_accuracy: 0.8344 - val_loss: 0.5955
Epoch 5/50
320/320 ━━━━━━━━━━━━━━━━━━━━ 13s 39ms/step - accuracy: 0.9832 - loss: 0.0524 - val_accuracy: 0.8293 - val_loss: 0.6983
Epoch 6/50
320/320 ━━━━━━━━━━━━━━━━━━━━ 13s 39ms/step - accuracy: 0.9891 - loss: 0.0358 - val_accuracy: 0.8348 - val_loss: 0.7707
Epoch 7/50
320/320 ━━━━━━━━━━━━━━━━━━━━ 21s 40ms/step - accuracy: 0.9925 - loss: 0.0262 - val_accuracy: 0.8297 - val_loss: 0.8450
Epoch 8/50
320/320 ━━━━━━━━━━━━━━━━━━━━ 12s 39ms/step - accuracy: 

In [93]:
import numpy as np

def predict_emotion(text, cv, ps, stopwords_list, lstm_model):
    # Apply cleaning
    text = clean_html(text)
    text = remove_special(text)

    # Tokenize and remove stopwords
    tokens = text.split()
    filtered_tokens = [i for i in tokens if i not in stopwords_list]

    # Stem words
    stemmed_tokens = [ps.stem(i) for i in filtered_tokens]

    # Join back
    processed_text = " ".join(stemmed_tokens)

    # Transform using CountVectorizer
    vectorized_text = cv.transform([processed_text]).toarray()

    # Reshape for LSTM input
    reshaped_text = vectorized_text.reshape(1, 1, vectorized_text.shape[1])

    # Predict using the LSTM model
    prediction = lstm_model.predict(reshaped_text)
    predicted_class = np.argmax(prediction, axis=1)[0]

    return predicted_class

# Example usage:
new_text = "I miss you so much"

# Assuming `cv`, `ps`, `stopwords.words('english')` (as stopwords_list), and `lstm_model` are already defined and trained/fitted
# You can also define a mapping for labels if you know them
emotion_labels = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

predicted_label_index = predict_emotion(new_text, cv, ps, stopwords.words('english'), lstm_model)
predicted_emotion = emotion_labels.get(predicted_label_index, 'Unknown')

print(f"The predicted emotion for the text \"{new_text}\" is: {predicted_emotion}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
The predicted emotion for the text "I miss you so much" is: sadness
